# Liveness ringan: Face Landmark + Challenge Kanan/Kiri

Notebook ini menggantikan pendekatan CNN anti-spoof CASIA-FASD dengan **challenge-response berbasis face landmarks**. Tidak ada training model liveness. MediaPipe Face Landmarker mendeteksi landmark wajah secara real-time, lalu notebook menghitung perubahan posisi relatif hidung terhadap garis mata untuk menentukan gerakan kepala kiri/kanan.

**Flow:** kamera/video → face landmarks → kalibrasi posisi netral → challenge `LEFT` / `RIGHT` → validasi gerakan temporal → `LIVE` → ambil frame terbaik untuk Face Recognition di backend.

Catatan: metode ini adalah **challenge-response liveness**, bukan PAD/CNN anti-spoof yang dapat menjamin semua replay attack tertolak. Untuk TA, uji dengan kamera target dan video replay yang relevan. MediaPipe Face Landmarker mendukung image, video, dan live stream serta menghasilkan 3D landmarks; modelnya dirancang untuk real-time/mobile.


In [ ]:
!pip -q install mediapipe opencv-python-headless matplotlib pandas


In [ ]:
import os, json, math, urllib.request
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp

SEED = 42
MODEL_PATH = Path("/content/face_landmarker.task")

# Challenge settings. Start with these values, then calibrate on your target phone.
CALIBRATION_FRAMES = 15
HOLD_FRAMES = 4
MAX_CHALLENGE_FRAMES = 45
MIN_FACE_PRESENCE = 0.5
MIN_TRACKING = 0.5
# Change this if the device/frontend uses mirrored frames.
MIRRORED_INPUT = False

# Proxy-yaw thresholds are relative to the user's neutral face.
# Larger absolute values mean a stronger head turn.
LEFT_DELTA = -0.10
RIGHT_DELTA = 0.10

print("MediaPipe:", mp.__version__)


## 1. Model Face Landmarker

MediaPipe Face Landmarker menyediakan face landmarks 3D dan dapat dijalankan pada live video. Kita hanya membutuhkan landmark; blendshape dan transformation matrix tidak diwajibkan untuk metode ini.


In [ ]:
# Official MediaPipe model bundle. If the URL changes, download face_landmarker.task manually
# and set MODEL_PATH to the downloaded file.
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task"
if not MODEL_PATH.exists():
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print("Model:", MODEL_PATH, MODEL_PATH.stat().st_size, "bytes")


In [ ]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path=str(MODEL_PATH))
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=MIN_FACE_PRESENCE,
    min_face_presence_confidence=MIN_FACE_PRESENCE,
    min_tracking_confidence=MIN_TRACKING,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
)
landmarker = vision.FaceLandmarker.create_from_options(options)
print("FaceLandmarker ready")


## 2. Hitung posisi relatif hidung

Kita tidak mengklasifikasikan gambar menjadi `real/spoof`. Sebaliknya, kita mengukur **perubahan pose kepala** terhadap posisi netral.

Landmark yang digunakan:
- `33` dan `263`: sudut luar mata kiri/kanan.
- `1`: titik hidung.

Nilai `yaw_proxy` dinormalisasi terhadap jarak kedua mata agar lebih stabil terhadap jarak wajah ke kamera.


In [ ]:
LEFT_EYE_OUTER = 33
RIGHT_EYE_OUTER = 263
NOSE_TIP = 1

def _landmark_xy(face_landmarks, idx):
    p = face_landmarks[idx]
    return np.array([p.x, p.y], dtype=np.float32)

def yaw_proxy(face_landmarks):
    left_eye = _landmark_xy(face_landmarks, LEFT_EYE_OUTER)
    right_eye = _landmark_xy(face_landmarks, RIGHT_EYE_OUTER)
    nose = _landmark_xy(face_landmarks, NOSE_TIP)
    eye_mid = (left_eye + right_eye) / 2.0
    eye_dist = float(np.linalg.norm(right_eye - left_eye))
    if eye_dist < 1e-6:
        return None
    return float((nose[0] - eye_mid[0]) / eye_dist)

def frame_yaw_proxy(frame_bgr, timestamp_ms):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect_for_video(mp_image, timestamp_ms)
    if not result.face_landmarks:
        return None
    return yaw_proxy(result.face_landmarks[0])


## 3. Kalibrasi + challenge kiri/kanan

Implementasi ini memakai **baseline netral**. User diminta melihat lurus selama beberapa frame, lalu sistem memberi challenge.

Contoh:

`RIGHT` → user menoleh kanan → nilai proxy naik melewati threshold → challenge selesai.

`LEFT` → user menoleh kiri → nilai proxy turun melewati threshold → challenge selesai.

Kita mensyaratkan beberapa frame berturut-turut agar satu frame noise tidak langsung dianggap berhasil.


In [ ]:
def calibrate_baseline(values):
    values = [v for v in values if v is not None]
    if len(values) < max(5, CALIBRATION_FRAMES // 2):
        raise ValueError("Frame wajah valid untuk kalibrasi terlalu sedikit")
    return float(np.median(values))

def challenge_target_passed(delta, target):
    if target == "LEFT":
        return delta <= LEFT_DELTA
    if target == "RIGHT":
        return delta >= RIGHT_DELTA
    raise ValueError("target harus LEFT atau RIGHT")

def evaluate_challenge(deltas, target, hold_frames=HOLD_FRAMES, max_frames=MAX_CHALLENGE_FRAMES):
    valid = [float(d) for d in deltas if d is not None]
    for i in range(max(0, len(valid) - hold_frames + 1)):
        window = valid[i:i+hold_frames]
        if all(challenge_target_passed(d, target) for d in window):
            return {"passed": True, "target": target, "frames_used": i + hold_frames, "peak_delta": float(max(window) if target == "RIGHT" else min(window))}
    return {"passed": False, "target": target, "frames_used": min(len(valid), max_frames), "peak_delta": None}


## 4. Uji pada video

Upload video pendek dari kamera HP dengan pola:
1. lihat lurus saat kalibrasi,
2. tunggu challenge,
3. lakukan gerakan kiri/kanan.

Video ini hanya untuk validasi algoritma. Di Flutter, logic yang sama sebaiknya dijalankan frame-by-frame pada kamera live agar video tidak perlu dikirim penuh ke backend.


In [ ]:
from google.colab import files

uploaded = files.upload()
VIDEO_PATH = next(iter(uploaded.keys()))
print("Video:", VIDEO_PATH)


In [ ]:
def analyze_video(video_path, target="RIGHT"):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    values = []
    frames = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        ts = int(frames * 1000.0 / fps)
        values.append(frame_yaw_proxy(frame, ts))
        frames += 1
    cap.release()
    if len(values) < CALIBRATION_FRAMES:
        raise ValueError("Video terlalu pendek untuk kalibrasi")
    baseline = calibrate_baseline(values[:CALIBRATION_FRAMES])
    deltas = [None if v is None else v - baseline for v in values[CALIBRATION_FRAMES:]]
    result = evaluate_challenge(deltas, target)
    result.update({"baseline": baseline, "frames_total": len(values), "valid_frames": int(sum(v is not None for v in values))})
    return result, values, deltas

# Ubah target menjadi LEFT jika video berisi gerakan kiri.
TARGET = "RIGHT"
result, raw_values, deltas = analyze_video(VIDEO_PATH, TARGET)
print(json.dumps(result, indent=2))


In [ ]:
# Visualisasi proxy yaw untuk membantu memilih threshold pada kamera target.
plt = __import__('matplotlib.pyplot', fromlist=['plt'])
plt.figure(figsize=(12,4))
plt.plot(raw_values, label='yaw_proxy')
plt.axhline(0, linestyle='--')
plt.axvline(CALIBRATION_FRAMES, linestyle=':')
plt.title(f'Face landmark yaw proxy — target {TARGET}')
plt.xlabel('Frame')
plt.ylabel('Normalized nose offset')
plt.legend()
plt.show()


## 5. Format logic yang dibawa ke backend

Tidak perlu menyimpan model CNN atau dataset liveness. Backend cukup menerima hasil challenge dari mobile **beserta metadata challenge** yang dibuat server agar replay lebih sulit. Contoh payload:

```json
{
  "challenge_id": "random-id",
  "challenge": ["RIGHT", "LEFT"],
  "passed": true,
  "duration_ms": 2100,
  "face_count_max": 1
}
```

Lebih aman jika server yang menentukan urutan challenge secara acak, kemudian mobile menjalankan Face Landmarker secara lokal. Setelah challenge berhasil, mobile mengirim satu/few frame terbaik untuk proses Face Recognition yang sudah ada (ArcFace/SVM atau centroid), lalu Laravel melakukan validasi GPS polygon.


In [ ]:
DEPLOY_CONFIG = {
    "method": "face_landmark_challenge_response",
    "landmarks": {"left_eye_outer": LEFT_EYE_OUTER, "right_eye_outer": RIGHT_EYE_OUTER, "nose_tip": NOSE_TIP},
    "calibration_frames": CALIBRATION_FRAMES,
    "hold_frames": HOLD_FRAMES,
    "max_challenge_frames": MAX_CHALLENGE_FRAMES,
    "left_delta": LEFT_DELTA,
    "right_delta": RIGHT_DELTA,
    "mirrored_input": MIRRORED_INPUT,
    "model": "MediaPipe Face Landmarker",
    "note": "Thresholds must be calibrated and validated on the target mobile camera."
}
Path('/content/face_landmark_liveness_config.json').write_text(json.dumps(DEPLOY_CONFIG, indent=2), encoding='utf-8')
print(json.dumps(DEPLOY_CONFIG, indent=2))


## Dataset untuk validasi (bukan training)

Karena metode ini **tidak melatih classifier liveness**, dataset tidak diperlukan untuk training. Dataset head-pose dapat dipakai untuk menguji apakah threshold kiri/kanan masuk akal sebelum diuji dengan kamera HP. Salah satu pilihan kecil adalah **Head Pose Annotations Dataset**: 2.790 foto dengan ground-truth pan/tilt, dengan pan dari -90° sampai +90°.

Pilihan lain adalah **AISL Head Orientation Database**, berisi 6.480 gambar dari 20 subjek dengan 36 sudut pan, 3 sudut tilt, dan 3 background. Untuk uji yang lebih besar, 300W-LP menyediakan sekitar 122 ribu citra sintetis dengan variasi pose sampai ±90° yaw.

Untuk TA, saya lebih menyarankan dataset tersebut hanya sebagai **uji pendukung**, lalu buat dataset kecil sendiri dari kamera HP target untuk validasi akhir karena domain kamera sangat menentukan.


## Kesimpulan

Notebook ini sengaja tidak menghasilkan `liveness_model.keras`. Artifact utamanya adalah `face_landmark_liveness_config.json` dan algoritma challenge-response. Implementasi production sebaiknya: **random challenge dari server → landmark lokal di Flutter → challenge berhasil → kirim frame terbaik → ArcFace/recognition → GPS polygon → presensi**.
